In [1]:
print("hi")

hi


In [2]:
from dotenv import load_dotenv

In [3]:
load_dotenv()

True

In [4]:
from langchain_groq import ChatGroq

In [5]:
llm=ChatGroq(model="deepseek-r1-distill-llama-70b")

In [6]:
llm.invoke("what is the  gpd of australia").content

"<think>\nOkay, so I need to figure out what the GDP of Australia is. I'm not exactly sure what GDP stands for, but I think it's related to economics. Maybe it's like the total money a country makes? I've heard people talk about GDP when discussing how well a country's economy is doing. \n\nI remember hearing that GDP stands for Gross Domestic Product. So, I guess it measures the total economic output of a country. But how exactly is it calculated? I think it includes things like consumption, investment, government spending, and exports minus imports. That formula might be important for understanding how they come up with the number.\n\nNow, focusing on Australia, I'm not sure what their current GDP is. I think it's a developed country with a strong economy, so maybe their GDP is pretty high. I wonder how it compares to other countries like the US or China. I also remember that Australia has significant industries like mining, agriculture, and services. Those sectors probably contribut

In [7]:
import os
from langchain_community.tools.tavily_search import TavilySearchResults
TAVILY_API_KEY=os.getenv("TAVILY_API_KEY")
search_tool=TavilySearchResults(tavily_api_key=TAVILY_API_KEY)

/var/folders/7z/1fnfdg6j7v11htdvh6ltq_yw0000gn/T/ipykernel_98020/156554669.py:4: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-tavily package and should be used instead. To use it run `pip install -U :class:`~langchain-tavily` and import as `from :class:`~langchain_tavily import TavilySearch``.
  search_tool=TavilySearchResults(tavily_api_key=TAVILY_API_KEY)


In [8]:
search_tool.invoke("What is the capital of France?")

[{'title': 'Paris facts: the capital of France in history',
  'url': 'https://home.adelphi.edu/~ca19535/page%204.html',
  'content': '|  |  |  |  |  |  |  |\n ---  ---  --- \n| Home | Spain | Sydney | San Francisco | Paris | Las Vegas | Maui |\n\n  \nParis, France\n\n  \n\n## Paris facts: Paris, the capital of France\n\nParis is the capital of France,\nthe largest country of Europe\nwith 550 000 km2 (65 millions inhabitants).\n\nParis has 2.234 million inhabitants\nend 2011. She is the core of Ile de France region (12 million\npeople). [...] Paris remained the capital of\nFrance until today, with one four year interruption. During\nGerman occupation (WW2 , 1940-1944), the capital of France was Vichy.\n\ngo to top\n\nReference: [...] ## Paris facts: the capital of France in history\n\nBefore Paris, the capital of France\nwas Lyon\n(under the Romans). Paris first became the capital of France in\n508 under King Clovis. After centuries with no unique capital of\nFrance, Paris retrieved its

In [9]:
my_code = """
x=10
y=x+10
print(y)
"""

In [14]:

from langchain_experimental.utilities import PythonREPL

In [15]:
repl =PythonREPL()

In [17]:
repl.run(my_code)

Python REPL can execute arbitrary code. Use with caution.


'20\n'

In [18]:
from typing import Annotated

In [19]:

from langchain_core.tools import tool

In [ ]:
@tool
def python_repl_tool(code: Annotated[str, "The python code to execute to generate your chart."]):
    """Use this to execute python code and do math. If you want to see the output of a value,
    you should print it out with `print(...)`. This is visible to the user."""
    
    try:
        result = repl.run(code)
    except BaseException as e:
        return f"Failed to execute. Error: {repr(e)}"
    
    result_str = f"Successfully executed:\n\`\`\`python\n{code}\n\`\`\`\nStdout: {result}"
    return result_str

In [20]:
members =["researcher","coder"]

In [21]:
members

['researcher', 'coder']

In [23]:
options =members+ ["FINISH"]

In [24]:
options

['researcher', 'coder', 'FINISH']

In [25]:
from typing import Literal

In [26]:
from typing_extensions import TypedDict

In [27]:
class Router(TypedDict):
    next:Literal['researcher', 'coder', 'FINISH']

In [28]:
from langgraph.graph import MessagesState,StateGraph,START, END

In [ ]:
	
class State(MessagesState):
    next:str

this is how my state will be looking like

In [50]:
state={"messages": ["hi"], "next": "research_agent"}

In [30]:
system_prompt = f""""
You are a supervisor, tasked with managing a conversation between the following workers: {members}. 
Given the following user request, respond with the worker to act next. 
Each worker will perform a task and respond with their results and status. 
When finished, respond with FINISH.
"""

In [31]:
# system_prompt = f""""
# You are a supervisor, tasked with managing a conversation between the following workers: {members}. 
# Given the following user request, respond with the worker to act next. 
# Each worker will perform a task and respond with their results and status. 
# When finished, respond with FINISH.
# **Strict Guidelines:**
# if there is any common messages like hi, hello, how are you, greetings etc then,respond with FINISH.
# ""

In [48]:
print(system_prompt)

"
You are a supervisor, tasked with managing a conversation between the following workers: ['researcher', 'coder']. 
Given the following user request, respond with the worker to act next. 
Each worker will perform a task and respond with their results and status. 
When finished, respond with FINISH.



In [51]:

messages = [{"role": "system", "content": system_prompt},] + state["messages"]

In [53]:
messages

[{'role': 'system',
  'content': '"\nYou are a supervisor, tasked with managing a conversation between the following workers: [\'researcher\', \'coder\']. \nGiven the following user request, respond with the worker to act next. \nEach worker will perform a task and respond with their results and status. \nWhen finished, respond with FINISH.\n'},
 'hi']

In [57]:
llm_with_structure_output = llm.with_structured_output(Router)

In [58]:
llm_with_structure_output.invoke(messages)

{'next': 'researcher'}

In [33]:
from langgraph.types import Command

In [59]:
def supervisor_agent(state:State) ->    Command[Literal['researcher', 'coder', '__end__']]:
    messages = [{'role':'system',"content":system_prompt},] + state[messages]
    llm_with_structure_output = llm.with_structured_output(Router)

    response = llm_with_structure_output.invoke(messages)


    goto= response["next"]
    print("**********BELOW IS MY GOTO***************")
    
    print(goto)

    if goto == 'FINISH':
        goto=END

    return Command(goto=goto, update={"next":goto})    

In [60]:
from langgraph.prebuilt import create_react_agent

In [61]:
from langchain_core.messages import AIMessage, HumanMessage

In [62]:
def researcher_agent(state:State)-> Command[Literal['supervior']]:
    research_agent = create_react_agent(llm, tools=[search_tool], prompt="You are a researcher. DO NOT do any math.")

    response = research_agent.invoke(state)

    return Command(goto="supervisor",
                   update={"messages":[HumanMessage(content=response["messages"][-1].content, name="researcher")]
                           
                           },)

In [ ]:
def coder_agent(state:State)-> Command[Literal['supervior']]:
    code_agent = create_react_agent(llm, tools=[python_repl_tool],"You are a coder. DO NOT do any research.")
    result = code_agent.invoke(state)
    return Command(
        update={
            "messages": [
                HumanMessage(content=result["messages"][-1].content, name="coder")
            ]
        },
        goto="supervisor",
    )
        

### This is my ochestration flow with langgraph

In [38]:
grah = StateGraph(State)

In [39]:
grah.add_node("supervisor", supervisor_agent)

In [40]:
grah.add_node("researcher",researcher_agent)

In [41]:
grah.add_node("coder",coder_agent)

In [42]:
grah.add_edge(START,"supervisor")

In [43]:
app = grah.compile()

ValueError: Found edge ending at unknown node `FINISH`